# z302 – Feature Engineering
**Grupo 3: Banegas - Marín - Mengoni - Rey**

Lee el parquet de z301 → construye lags → normaliza → agrega deltas → elige target explícito → detecta leakage → escribe `dataset_fe.parquet`.

| Palanca | Opciones | Default |
|---------|----------|---------|
| `t0` | período de corte | 201910 |
| `max_lags` | historia a incluir | 13 |
| `normalizacion` | recta / zscore / minmax / media | recta |
| `tipo_target` | nivel / delta | nivel |

In [ ]:
import yaml, time
from pathlib import Path
import duckdb
import numpy as np
from google.cloud import storage as gcs

with open('../pipe_py/config.yaml') as f:
    CFG = yaml.safe_load(f)

# --- palancas ---
CFG['fe']['t0']           = 201910
CFG['fe']['max_lags']     = 13
CFG['fe']['tipo_target']  = 'nivel'   # nivel | delta
CFG['fe']['agregar_deltas'] = True
# ----------------

fe = CFG['fe']
pr = CFG['preproc']
grupo = f"{pr['group_mode']}_{pr['missing_strategy']}_{pr['densify_strategy']}"
ruta_preproc = Path(CFG['paths']['preproc_out']) / f'preprocesado_{grupo}.parquet'
print(f'Leyendo: {ruta_preproc}')

In [ ]:
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs")
con.execute(f"CREATE TABLE preproc AS SELECT * FROM read_parquet('{ruta_preproc}')")
n = con.execute('SELECT COUNT(*) FROM preproc').fetchone()[0]
print(f'Filas leídas: {n:,}')
con.execute('DESCRIBE preproc').df()

In [ ]:
# Función para restar N meses
def periodo_menos_n(t0, n):
    anio, mes = t0//100, t0%100
    mes -= n
    while mes <= 0:
        mes += 12; anio -= 1
    return anio*100 + mes

t0_val  = fe['t0']
max_lags = fe['max_lags']
periodos = [periodo_menos_n(t0_val, k) for k in range(max_lags+1)]
id_col = 'Agrupacion_ID'

print('Períodos que van a quedar como lags:', periodos)

In [ ]:
# Construcción de lags (pivot)
pivot_cols = ', '.join(
    f"MAX(CASE WHEN periodo = {p} THEN tn ELSE NULL END) AS tn{k}"
    for k, p in enumerate(periodos)
)
con.execute(f"""
    CREATE OR REPLACE TABLE lags AS
    SELECT {id_col}, {pivot_cols}
    FROM preproc GROUP BY {id_col}
""")
con.execute('SELECT * FROM lags LIMIT 3').df()

In [ ]:
# Normalización recta (min-max por fila)
tn_cols = [f'tn{k}' for k in range(max_lags+1)]
min_e = f"LEAST({', '.join(f'COALESCE(tn{k},0)' for k in range(max_lags+1))})"
max_e = f"GREATEST({', '.join(f'COALESCE(tn{k},0)' for k in range(max_lags+1))})"
norm_cols = ', '.join(
    f"CASE WHEN ({max_e} - {min_e}) = 0 THEN 0.0 "
    f"ELSE (COALESCE(tn{k},0) - {min_e}) / ({max_e} - {min_e}) END AS tn{k}_norm"
    for k in range(max_lags+1)
)
con.execute(f"""
    CREATE OR REPLACE TABLE normalizado AS
    SELECT *, {min_e} AS B0, ({max_e} - {min_e}) AS B1, {norm_cols}
    FROM lags
""")
print('Normalización OK')

In [ ]:
# Deltas
salto = fe['salto_delta']
if fe['agregar_deltas']:
    delta_cols = ', '.join(
        f'(tn{k}_norm - tn{k+salto}_norm) AS tn{k}_delta'
        for k in range(max_lags+1-salto)
    )
    con.execute(f"CREATE OR REPLACE TABLE con_deltas AS SELECT *, {delta_cols} FROM normalizado")
else:
    con.execute("CREATE OR REPLACE TABLE con_deltas AS SELECT * FROM normalizado")
print('Deltas:', fe['agregar_deltas'])

In [ ]:
# Target explícito (una sola columna 'target')
tipo_target = fe['tipo_target']
target_expr = 'tn0_norm' if tipo_target == 'nivel' else 'tn0_delta'
con.execute(f"""
    CREATE OR REPLACE TABLE dataset_fe AS
    SELECT *, {target_expr} AS target
    FROM con_deltas
""")
print(f'Target: {target_expr} → columna "target"')

In [ ]:
# Detección de leakage
lk = CFG['leakage']
excluir_fijas = set(fe['cols_excluir_leakage'] + ['target', 'Agrupacion_ID', 'B0', 'B1'])
todas = [r[0] for r in con.execute('DESCRIBE dataset_fe').fetchall() if r[0] not in excluir_fijas]

leakage_cols = []
for col in todas:
    try:
        corr = con.execute(f'SELECT ABS(CORR({col}, target)) FROM dataset_fe').fetchone()[0]
        if corr and corr >= lk['umbral_correlacion']:
            print(f'  [leakage] {col}  corr={corr:.4f}')
            leakage_cols.append(col)
    except: pass

print(f'Columnas con leakage: {leakage_cols if leakage_cols else "ninguna"}')

In [ ]:
# Resumen del dataset final
n_filas = con.execute('SELECT COUNT(*) FROM dataset_fe').fetchone()[0]
schema  = con.execute('DESCRIBE dataset_fe').df()
print(f'dataset_fe: {n_filas:,} filas x {len(schema)} columnas')
schema

In [ ]:
# Estadísticas del target
con.execute('SELECT MIN(target), MAX(target), AVG(target), STDDEV(target) FROM dataset_fe').df()

In [ ]:
# Guardar parquet
ruta_out = Path(CFG['paths']['fe_out']) / 'dataset_fe.parquet'
ruta_out.parent.mkdir(parents=True, exist_ok=True)
con.execute(f"COPY dataset_fe TO '{ruta_out}' (FORMAT PARQUET, COMPRESSION SNAPPY)")
print(f'Guardado → {ruta_out}')

# Subir a GCS
g = CFG['gcs']
client = gcs.Client()
bucket = client.bucket(g['bucket'])
blob = bucket.blob(f"{g['prefix_fe']}/dataset_fe.parquet")
blob.upload_from_filename(str(ruta_out))
print(f"Subido → gs://{g['bucket']}/{g['prefix_fe']}/dataset_fe.parquet")